# #9 Memorias MoI: compuerta de nombre completo (`Qwen/Qwen3.5-0.8B`, 1000 personas) — Colab L4

La corrida anterior (`moi_memory_colab.ipynb`) mostró que `rome_gated` le pasa el dato a **otra persona con el mismo
apellido** (`nb_last` 93% / 62% / 34% en 10 / 100 / 1000). La llave está en el último token del nombre = el apellido.

Nuevo brazo **`rome_fullname`**: la memoria solo se activa si el apellido coincide **y** su llave de nombre de pila
coincidió en los 6 tokens anteriores.

* **No reescribe memorias:** copia `progress.pt` de la corrida anterior (mismas opciones) y salta directo a evaluar
  (~15 min, casi todo es la covarianza y la evaluación).
* Brazos: `base, ctx, rome_gated, rome_fullname`.
* **Resultado final:** `MyDrive/wsi_runs/FINALES/moi_memory_fullname_qwen35-08b.json`. Luego el runtime se apaga solo.


In [ ]:
import os, json, shutil, subprocess, datetime
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "moi_memory_fullname_qwen35-08b"
REPO = "/content/wsi"
BASE = "/content/qwen-base"
from google.colab import drive
drive.mount("/content/drive")
PREV = "/content/drive/MyDrive/wsi_runs/moi_memory_qwen35-08b"   # corrida anterior: tiene progress.pt
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True)
if not os.path.exists(f"{DRIVE}/progress.pt"):
    assert os.path.exists(f"{PREV}/progress.pt"), f"falta {PREV}/progress.pt (sin él reescribe las 1000 memorias, ~75 min)"
    shutil.copy(f"{PREV}/progress.pt", f"{DRIVE}/progress.pt")
print("ya en Drive:", sorted(os.listdir(DRIVE)))


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U transformers
!pip install -q flash-linear-attention || echo "fla opcional"
import transformers, torch; print("transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))


In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
# mismas opciones que la corrida anterior (si no, ignora progress.pt) + solo los brazos que importan
OUT = f"{DRIVE}/results_moi_memory_fullname.json"
!python -u {REPO}/scripts/moi_memory.py --model {BASE} --dtype fp32 --tf32 --write-batch 16 --progress {DRIVE}/progress.pt --people 10,100,1000 --eval-people 50 --layers 3,5,7,9,11 --cov-tokens 100000 --ppl-tokens 32768 --tok-budget 8192 --arms base,ctx,rome_gated,rome_fullname --no-restart-check --out {OUT} 2>&1 | grep --line-buffered -v -E "Loading weights|fast path|Warning"


In [ ]:
# resultado final: a FINALES solo si la corrida terminó
res = json.load(open(OUT))
assert res.get("complete"), "la corrida no terminó: vuelve a correr la celda de arriba (sigue donde iba)"
res["_final"] = {"complete": True, "commit": COMMIT, "notebook": "moi_memory_fullname_colab.ipynb",
                 "gpu": torch.cuda.get_device_name(0),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1, ensure_ascii=False)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)
for N, sc in res["scales"].items():
    for arm, a in sc["arms"].items():
        p = a["probes"]
        print(N, arm.ljust(10), {k: p[k]["choice"] for k in ("write", "dev", "para", "es", "chat", "hop", "nb_first", "nb_last") if k in p},
              "fired nbL", p["nb_last"].get("own_fired"), "dev", p["dev"].get("own_fired"),
              "ppl", a.get("ppl", {}).get("ppl"), "caps", a.get("capitals_retained"))


In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
